<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.2-cloud-run-deploy/notebooks/GCP_Capstone_7.2_CloudRunDeploy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.2 Deploy to Cloud Run with IAM — Scale-to-Zero, MCP Toolbox
**Netsetos GenAI Engineering — GCP Capstone**


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
REGION = 'us-central1'

!gcloud config set project $PROJECT_ID
!gcloud config set run/region $REGION
print(f'Project: {PROJECT_ID}')


## Cell 1: Enable APIs


In [ ]:
!gcloud services enable run.googleapis.com artifactregistry.googleapis.com cloudbuild.googleapis.com secretmanager.googleapis.com


## Cell 2: Create Project Files


In [ ]:
import os
os.makedirs('documind-mcp', exist_ok=True)
with open('documind-mcp/pyproject.toml', 'w') as f:
    f.write('[project]\nname = "documind-mcp-server"\nversion = "0.1.0"\nrequires-python = ">=3.10"\ndependencies = ["fastmcp"]\n')
with open('documind-mcp/Dockerfile', 'w') as f:
    f.write('FROM python:3.13-slim\nCOPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/\nCOPY . /app\nWORKDIR /app\nENV PYTHONUNBUFFERED=1\nRUN uv sync\nEXPOSE $PORT\nCMD ["uv", "run", "server.py"]\n')
print(f'Files: {os.listdir("documind-mcp")}')


## Cell 3: Create Server


In [ ]:
server = '''import asyncio, os, logging
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from typing import Literal
logging.basicConfig(level=logging.INFO)
mcp = FastMCP("DocuMind")
DOCS = [{"id":"doc-001","title":"Q4 Financial Report","cat":"financial","pages":24},
        {"id":"doc-002","title":"Engineering Design","cat":"technical","pages":18},
        {"id":"doc-003","title":"Employee Handbook","cat":"hr","pages":45},
        {"id":"doc-004","title":"Marketing Strategy","cat":"marketing","pages":12}]
@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search documents by keyword."""
    if not query.strip(): raise ToolError("Query empty")
    return [{"id":d["id"],"title":d["title"]} for d in DOCS if query.lower() in d["title"].lower()][:max_results]
@mcp.tool
def calculate_cost(page_count: int, processing_type: Literal["standard","premium","enterprise"]="standard") -> dict:
    """Calculate processing cost."""
    if page_count<=0: raise ToolError("Positive pages required")
    rates={"standard":0.01,"premium":0.03,"enterprise":0.05}
    return {"cost":round(rates[processing_type]*page_count,2)}
@mcp.tool
def get_stats() -> dict:
    """Get repository stats."""
    return {"total":len(DOCS),"pages":sum(d["pages"] for d in DOCS)}
@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document."""
    if not title.strip(): raise ToolError("Title required")
    kws={"financial":["revenue","budget"],"technical":["api","system"],"hr":["employee"],"marketing":["campaign"]}
    text=(title+" "+content).lower()
    scores={c:sum(1 for k in ws if k in text) for c,ws in kws.items()}
    return {"category":max(scores,key=scores.get)}
if __name__=="__main__":
    port=int(os.getenv("PORT",8080))
    asyncio.run(mcp.run_async(transport="streamable-http",host="0.0.0.0",port=port))
'''
with open('documind-mcp/server.py','w') as f: f.write(server)
print('server.py created')


## Cell 4: Deploy


In [ ]:
%cd documind-mcp
!gcloud run deploy documind-mcp-server --no-allow-unauthenticated --region=$REGION --source .
!gcloud run services describe documind-mcp-server --region=$REGION --format='value(status.url)'


## Cell 5: Test


In [ ]:
# Get URL and token
import subprocess
url = subprocess.check_output('gcloud run services describe documind-mcp-server --region=us-central1 --format="value(status.url)"', shell=True).decode().strip()
print(f'MCP endpoint: {url}/mcp')
print(f'Test: gcloud run services proxy documind-mcp-server --region=us-central1')


## Cell 6: Toolbox Config


In [ ]:
toolbox = '''kind: source\nname: documind_db\ntype: cloud-sql-postgres\nproject: PROJECT_ID\nregion: us-central1\ninstance: documind-instance\ndatabase: documents\n---\nkind: tool\nname: search-docs-db\ntype: postgres-sql\nsource: documind_db\ndescription: Search documents by keyword.\nparameters:\n  - name: query\n    type: string\nstatement: SELECT id,title FROM documents WHERE title ILIKE $1 LIMIT 10;\n'''
with open('tools.yaml','w') as f: f.write(toolbox)
print('tools.yaml created')


## ✅ Lesson 7.2 Complete!
- ✅ Cloud Run deployment with IAM
- ✅ Scale-to-zero ($0 idle)
- ✅ Secret Manager
- ✅ MCP Toolbox config

**Next: 7.3 — Connect Gemini to Cloud Run MCP**
